# ChEMBL fragment space analysis
Henrik Bermanis 09/10/2026

Ran on ChEMBL 37 release 2026-05-29 07:35

## Imports and settings

In [ ]:
import csv
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors, rdFingerprintGenerator

# ChEMBL trips loads of parser warnings in the passes below.
RDLogger.DisableLog("rdApp.*")

DB = Path("Data/chembl_37/chembl_37_sqlite/chembl_37.db")
FRAGMENTS = Path("Data/chembl_37_fragments.csv")
EMBEDDING = Path("Data/chembl_37_fragments_umap.csv")
FIGURE_STEM = Path("Artifacts/chembl_37_fragments_umap")

# Just to make sure memory is more or less constant 
BATCH = 50000

# Strip stereochemistry while the frags file is written, so stereoisomers collapse
# into one row instead of occupying one row (and one UMAP point) each.
FLATTEN = True

# Same consts as in the old script
RADIUS = 2          
N_BITS = 2048
N_NEIGHBORS = 15
MIN_DIST = 0.1
N_JOBS = 24 # All Studio's power cores

print(DB, f"{DB.stat().st_size / 1e9:.1f} GB" if DB.exists() else "MISSING")
print(f"FLATTEN = {FLATTEN}" + ("  (stereochemistry stripped)" if FLATTEN else ""))

# Extract the fragments

Use SQL against ChEMBL mol properties so only the frags reach rdkit and the massive file is never loaded in all. 
<br><br>
Change the query to get different subsets of frags. NOTE: MW is judged by the freebase weight to accomodate salts. 

In [ ]:
QUERY = """
    SELECT md.chembl_id, cs.canonical_smiles, cp.mw_freebase
    FROM compound_structures AS cs
    JOIN compound_properties AS cp USING (molregno)
    JOIN molecule_dictionary AS md USING (molregno)
    WHERE cs.canonical_smiles IS NOT NULL
      AND cp.mw_freebase <= 400
      AND cp.alogp <= 3
      AND cp.hbd <= 3
      AND cp.hba <= 6
      AND cp.rtb <= 5
"""

# Read-only URI, so nothing can modify the 30 GB release.
connection = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

# Return frag proportion
n_smiles = connection.execute("SELECT COUNT(*) FROM compound_structures WHERE canonical_smiles IS NOT NULL").fetchone()[0]
n_frags = connection.execute(f"SELECT COUNT(*) FROM ({QUERY})").fetchone()[0]

print(f"structures with SMILES: {n_smiles:,}")
print(f"rule-of-three rows:     {n_frags:,} ({100 * n_frags / n_smiles:.1f}%)")

## Clean mols, stream the release, and write the frags file

Parse, drop the counter-ion, canonicalise. Batched fetch, streaming write, dedup on canonical SMILES.

In [ ]:
def clean(smiles, mw_freebase):
    """
    Parse, drop the counter-ion if present, and return a canonical smiles. 
    Keeps the freebase frag to avoid instances where counter-ion is heavier
    and thus kept, e.g., pamoates, tosylates, picrates, etc.  
    With FLATTEN on, stereochem is dropped before canonicalising, so the dedup
    below folds stereoisomers into a single row. Isotope labels survive, so
    isotopologues stay separate either way.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    if "." in smiles:
        pieces = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
        mol = min(pieces, key=lambda piece: abs(Descriptors.MolWt(piece) - mw_freebase))
    if FLATTEN:
        Chem.RemoveStereochemistry(mol)  # works in place
    return Chem.MolToSmiles(mol)

cursor = connection.execute(QUERY)
seen = set()
n_rows = 0

FRAGMENTS.parent.mkdir(parents=True, exist_ok=True)
with FRAGMENTS.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["chembl_id", "smiles"])

    while rows := cursor.fetchmany(BATCH):
        n_rows += len(rows)
        
        for chembl_id, smiles, mw_freebase in rows:
            canonical = clean(smiles, mw_freebase)
            if canonical and canonical not in seen:
                seen.add(canonical)
                writer.writerow([chembl_id, canonical])
        print(f"  {n_rows:,} rows scanned, {len(seen):,} fragments kept", flush=True)

connection.close()
print(f"\n{n_rows:,} rows -> {len(seen):,} unique fragments"
      f"{' (stereochemistry stripped)' if FLATTEN else ''} -> {FRAGMENTS}")

# Filter Diagnostics

What's filtered out and what's kep. Read-only and adjacent to the workflow so nothing here feeds the fprints below.
Every check prints a summary and keeps its dataframe in `DIAGNOSTIC_FRAMES`, all of which get written to `Data/diagnostics/` in the last cell.

In [ ]:
import re
from rdkit.Chem import Crippen, Lipinski

DIAGNOSTICS = Path("Data/diagnostics")
DIAGNOSTIC_FRAMES = {}  # name -> frame, all written out in the last cell

# Open a fresh read-only connection.
diagnostics_connection = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

# Thresholds are read back out of QUERY itself, so these checks can't drift.
LIMITS = {name: float(limit) for name, limit in re.findall(r"cp\.(\w+)\s*<=\s*([\d.]+)", QUERY)}
MW_MAX = LIMITS.pop("mw_freebase")

# Rows that clear MW on its own: the denominator for the next two checks.
WITHIN_MW = f"""
    FROM compound_structures AS cs
    JOIN compound_properties AS cp USING (molregno)
    JOIN molecule_dictionary AS md USING (molregno)
    WHERE cs.canonical_smiles IS NOT NULL
      AND cp.mw_freebase <= {MW_MAX:g}
"""
n_within_mw = diagnostics_connection.execute(f"SELECT COUNT(*) {WITHIN_MW}").fetchone()[0]

# Re-read the file so this section stands on its own, and parse once for reuse.
fragments = pd.read_csv(FRAGMENTS)
mols = fragments["smiles"].map(Chem.MolFromSmiles)

print("QUERY filters on: " + ", ".join([f"mw_freebase <= {MW_MAX:g}"]
                                       + [f"{name} <= {limit:g}" for name, limit in LIMITS.items()]))
print(f"rows within MW alone: {n_within_mw:,}")
print(f"fragments on disk:    {len(fragments):,} (unparsable: {int(mols.isna().sum())})")

### 1. Within the MW limit, rejected by something else

A property that is `NULL` fails the comparison in `QUERY` too, so those are counted here separate.

In [ ]:
# Any property over its limit, or missing altogether.
why_rejected = " OR ".join(f"cp.{name} IS NULL OR cp.{name} > {limit:g}" for name, limit in LIMITS.items())
rejected = pd.read_sql_query(f"""SELECT md.chembl_id, cs.canonical_smiles AS smiles, cp.mw_freebase, 
                            {", ".join("cp." + name for name in LIMITS)} {WITHIN_MW} AND ({why_rejected})""", 
                            diagnostics_connection,)

over = pd.DataFrame({name: rejected[name] > limit for name, limit in LIMITS.items()})
missing = pd.DataFrame({name: rejected[name].isna() for name in LIMITS})
culprits = over | missing
rejected["n_reasons"] = culprits.sum(axis=1)
rejected["reasons"] = [" + ".join(culprits.columns[row]) for row in culprits.to_numpy()]
DIAGNOSTIC_FRAMES["rejected_by_property"] = rejected

print(f"{len(rejected):,} of the {n_within_mw:,} rows within MW "
      f"({100 * len(rejected) / n_within_mw:.1f}%) are rejected by another property")

print(f"all four properties NULL (elements, inorganics): {int(missing.all(axis=1).sum()):,}")
display(pd.DataFrame({"over the limit": over.sum(), "missing (NULL)": missing.sum()}))

# Eyeball the discards: lists the reason for discarding below as well. 
sample = rejected.sample(8, random_state=0)
sample_mols = [Chem.MolFromSmiles(smiles) for smiles in sample["smiles"]]
display(Draw.MolsToGridImage([mol for mol in sample_mols if mol is not None], molsPerRow=4, subImgSize=(220, 180),
                             legends=[reasons for mol, reasons in zip(sample_mols, sample["reasons"]) if mol is not None],))

### 2. Stereoisomers that end up as separate rows

`Chem.MolToSmiles` keeps stereochem, and dedup is on that string, so every
stereoisomer ChEMBL registers separately is its own row and its own point in the
UMAP. Stripping stereo from a copy of each molecule shows which rows are only
separated by it.

With `FLATTEN` on this ran upstream already, so the check still runs but finds nothing.

In [ ]:
def flatten(mol):
    """Canonical SMILES with stereo stripped. Isotope labels survive,
    so isotopologues aren't mistaken for stereoisomers."""
    flat = Chem.Mol(mol)  # RemoveStereochemistry works in place
    Chem.RemoveStereochemistry(flat)
    return Chem.MolToSmiles(flat)

flat_smiles = mols.map(flatten)

# Group the rows that share a skeleton; a group of one is just a molecule.
repeated = flat_smiles.duplicated(keep=False)
# The line below could've been split up, but you know... 
stereo_sets = (fragments[repeated].assign(flat_smiles=flat_smiles[repeated]).groupby("flat_smiles")
               .agg(n_rows=("smiles", "size"), smiles=("smiles", ";".join), chembl_ids=("chembl_id", ";".join)).sort_values("n_rows", ascending=False))
DIAGNOSTIC_FRAMES["stereoisomer_sets"] = stereo_sets

print(f"rows carrying stereochemistry:      {int((flat_smiles != fragments['smiles']).sum()):,}")
print(f"distinct skeletons ignoring stereo: {flat_smiles.nunique():,} of {len(fragments):,} rows")

# The worst offender, drawn out. With FLATTEN on there is nothing to draw.
if stereo_sets.empty:
    print("no rows differ only by stereochemistry"
          + (" (FLATTEN is on, so they were folded together upstream)" if FLATTEN else ""))
else:
    largest = stereo_sets.index[0]
    members = stereo_sets.loc[largest, "smiles"].split(";")
    print(f"largest set ({len(members)} rows): {largest}")
    display(Draw.MolsToGridImage([Chem.MolFromSmiles(smiles) for smiles in members], molsPerRow=4, subImgSize=(200, 170),
                                 legends=stereo_sets.loc[largest, "chembl_ids"].split(";"),))

### 3. Frags too small to mean anything

The rule have no floor, so methane, water and bare halide ions clear it.<br>
Two overlapping ways of catching them:
1. Three heavy atoms or fewer, and no carbon at all. 
2. Counter-ions that survived desalting because ChEMBL registered them as a separate entry.

In [ ]:
TINY_HEAVY_ATOMS = 3

heavy_atoms = mols.map(lambda mol: mol.GetNumHeavyAtoms())
mol_weight = mols.map(Descriptors.MolWt)
has_carbon = mols.map(lambda mol: any(atom.GetSymbol() == "C" for atom in mol.GetAtoms()))

tiny = (fragments.assign(heavy_atoms=heavy_atoms, mol_weight=mol_weight.round(2), has_carbon=has_carbon).loc[(heavy_atoms <= TINY_HEAVY_ATOMS) | ~has_carbon]
        .sort_values(["heavy_atoms", "mol_weight"]))
DIAGNOSTIC_FRAMES["tiny_fragments"] = tiny

# Visual check of the tiny frags. 
display(Draw.MolsToGridImage([Chem.MolFromSmiles(smiles) for smiles in tiny["smiles"].head(24)], molsPerRow=6, subImgSize=(150, 130),
                             legends=tiny["smiles"].head(24).tolist(),))

### Write the frames out

One CSV per check, in `Data/diagnostics/`. List columns are `;`-joined so they survive the round trip.

In [ ]:
DIAGNOSTICS.mkdir(parents=True, exist_ok=True)

manifest = []
for name, frame_out in DIAGNOSTIC_FRAMES.items():
    path = DIAGNOSTICS / f"{name}.csv"
    # Keep the index only where it carries something (a SMILES, a property name)
    # rather than a row number.
    frame_out.to_csv(path, index=not pd.api.types.is_integer_dtype(frame_out.index))
    manifest.append({"frame": name,
                     "rows": len(frame_out),
                     "columns": frame_out.shape[1],
                     "file": str(path)})

display(pd.DataFrame(manifest).set_index("frame"))

# Get Morgan fprints

Unparsable rows drop out and `kept` maps the matrix back onto the frame.

In [ ]:
frame = pd.read_csv(FRAGMENTS)
generator = rdFingerprintGenerator.GetMorganGenerator(radius=RADIUS, fpSize=N_BITS)

smiles_list = frame["smiles"].tolist()
matrix = np.zeros((len(smiles_list), N_BITS), dtype=np.uint8) # Expects a uint8
kept = []

for i, smi in enumerate(smiles_list):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        matrix[len(kept)] = generator.GetFingerprintAsNumPy(mol)
        kept.append(i)
    else:
        print(f"  {i:,} {smi} -> None")

matrix = matrix[: len(kept)]
frame = frame.iloc[kept].reset_index(drop=True)
bits = matrix.sum(axis=1)

# Quick sanity checks for dims, emtpy fingerprints, and bits never used
print(f"fingerprint matrix: {matrix.shape} ({matrix.nbytes / 1e6:.0f} MB)")
print("all-zero fingerprints:", int((bits == 0).sum()))
print("bits never used:", int((matrix.sum(axis=0) == 0).sum()), "of", N_BITS)

## Fit UMAP

Fits UMAP, saves the coords, and prints inspection info. 

In [ ]:
import umap 

# This does not include random_state=42, because seeding makes UMAP fall back to
# a single-threaded implementation. 
reducer = umap.UMAP(n_neighbors=N_NEIGHBORS,
                    min_dist=MIN_DIST,
                    n_components=2,
                    metric="jaccard",
                    n_jobs=N_JOBS,
                    #random_state=42,
                    verbose=True,)
embedding = reducer.fit_transform(matrix)

# Save coords and print some info 
print("embedding:", embedding.shape)
frame["umap_1"] = embedding[:, 0]
frame["umap_2"] = embedding[:, 1]
frame.to_csv(EMBEDDING, index=False)
print(EMBEDDING)
frame.head()

## Plots
Both rendered in PNG as well as SVG for resolution. 

### Denisty plot

Binned counts on a log colour scale. UMAP axes carry no units, so none given. 

In [ ]:
# Colour gradients didn't render neatly, so falling back to a simple blue ramp.
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
GREY = "#52514e"

TITLE = (f"ChEMBL fragment space: UMAP of Morgan fingerprints\n"
         f"{len(frame):,} fragments, radius {RADIUS}, {N_BITS} bits, Jaccard metric")

figure, axes = plt.subplots(figsize=(10, 9))
mesh = axes.hexbin(embedding[:, 0], embedding[:, 1], gridsize=200,
                   cmap=LinearSegmentedColormap.from_list("density", BLUE_RAMP),
                   norm=LogNorm(vmin=1), mincnt=1, linewidths=0.0,)

bar = figure.colorbar(mesh, ax=axes, fraction=0.045, pad=0.02)
bar.set_label("fragments per bin (log scale)", color=GREY)

axes.set_title(TITLE, fontsize=13, loc="left", pad=12)
axes.set_xlabel("UMAP 1", color=GREY)
axes.set_ylabel("UMAP 2", color=GREY)
axes.set_xticks([]); axes.set_yticks([])
axes.set_aspect("equal", adjustable="datalim")
figure.tight_layout()

FIGURE_STEM.parent.mkdir(parents=True, exist_ok=True)
for extension in ("svg", "png"):
    figure.savefig(FIGURE_STEM.with_name(f"Density_{FIGURE_STEM.name}.{extension}"),
                   dpi=200, bbox_inches="tight")
plt.show()

### Scatter plot

Rendered large, so individual frags stay visible instead of saturating into blobs. 

In [ ]:
figure, axes = plt.subplots(figsize=(20, 20))
axes.scatter(embedding[:, 0], embedding[:, 1], s=1.1, c="#2a78d6", alpha=0.9, linewidths=0.0)

axes.set_title(TITLE, fontsize=26, loc="left", pad=24)
axes.set_xlabel("UMAP 1", color=GREY, fontsize=20)
axes.set_ylabel("UMAP 2", color=GREY, fontsize=20)
axes.set_xticks([]); axes.set_yticks([])
axes.set_aspect("equal", adjustable="datalim")
figure.tight_layout()

for extension in ("svg", "png"):
    figure.savefig(FIGURE_STEM.with_name(f"Scatter_{FIGURE_STEM.name}.{extension}"),
                   dpi=200, bbox_inches="tight")
plt.show()

### Scaffold representation

Murcko scaffold per fragment, then one end of the representation ranking. 

`SCAFFOLD_END` picks the `"bottom"` for the least represented scaffolds or `"top"` for the most represented. The size of the cut is either a count or a fraction:
- `SCAFFOLD_NUMBER` takes exactly that many scaffolds from the chosen end. `SCAFFOLD_FRACTION` has to be `None` for this (the cell raises otherwise), and `SCAFFOLD_BASIS` no longer means anything, so it warns if that's still set to `"fragments"`.
- `SCAFFOLD_FRACTION` takes n%, with `SCAFFOLD_BASIS` deciding n% *of what*. Either way bottom 5%, bottom 1%, top 5% and top 1% are just these knobs.

`SCAFFOLD_BASIS` matters because the two readings of "5%" come apart at the two ends:
- `"fragments"` takes whole scaffolds from the chosen end until 5% of the frags are accounted for.
- `"scaffolds"` takes 5% of the distinct scaffolds instead. At the top end that's the reading that stays useful, since the single most common scaffold already carries far more than 5% of the frags on its own (benzene) -- under the `"fragments"` basis a top-end cut just collapses onto that one scaffold (at least one is always kept, so it never plots nothing).

Two more notes: 
1. Around 2/3 of the scaffolds appear exactly once, so a bottom-end cut lands inside that tie. *Which* singletons fall inside it is arbitrary (deterministic, but arbitrary) -- the red points are a representative sample of the sparse tail, not a privileged subset of it. 
2. Fargs with no ring have no Murcko scaffold at all, so they can never be selected; they are counted separately below.

In [ ]:
import warnings
from rdkit.Chem.Scaffolds import MurckoScaffold

SCAFFOLD_END = "top"       # "bottom" = least represented scaffolds, "top" = most represented
SCAFFOLD_NUMBER = 5        # take exactly this many scaffolds; None -> cut by SCAFFOLD_FRACTION instead
SCAFFOLD_FRACTION = None      # the n% to take: 0.05, 0.01, ...; set to None when cutting by SCAFFOLD_NUMBER
SCAFFOLD_BASIS = "scaffolds"  # "fragments" = n% of the plotted frags, "scaffolds" = n% of the distinct scaffolds
RED = "#d1344a"

# The two cuts answer the same question, so setting both would be ambiguous.
if SCAFFOLD_NUMBER is not None and SCAFFOLD_FRACTION is not None:
    raise ValueError("SCAFFOLD_NUMBER and SCAFFOLD_FRACTION are two ways to cut: set one of them to None.")
if SCAFFOLD_NUMBER is None and SCAFFOLD_FRACTION is None:
    raise ValueError("nothing to select: set one of SCAFFOLD_NUMBER or SCAFFOLD_FRACTION")
if SCAFFOLD_NUMBER is not None and SCAFFOLD_BASIS == "fragments":
    # This is a warning since that is legitimate, but likely not what is intended.
    warnings.warn("SCAFFOLD_BASIS = 'fragments' is ignored while SCAFFOLD_NUMBER is set: a count always cuts on scaffolds", stacklevel=2)

by_count = SCAFFOLD_NUMBER is not None
sparse_end = SCAFFOLD_END == "bottom"
END_WORD = "least" if sparse_end else "most"
SELECTION_LABEL = f"{SCAFFOLD_END}{SCAFFOLD_NUMBER:d}" if by_count else f"{SCAFFOLD_END}{100 * SCAFFOLD_FRACTION:g}pct"
SELECTION_WORDS = f"{SCAFFOLD_NUMBER:,} Murcko scaffolds" if by_count else f"{100 * SCAFFOLD_FRACTION:g}% of Murcko scaffolds"

# Murcko = ring systems plus the linkers between them, so an acyclic frag comes back as an empty str.
scaffolds = frame["smiles"].map(lambda smiles: MurckoScaffold.MurckoScaffoldSmiles(mol=Chem.MolFromSmiles(smiles))) 
acyclic = scaffolds == ""

# Chosen end first, ties broken on the SMILES so the cut is reproducible.
ranked = (scaffolds[~acyclic].value_counts().rename_axis("scaffold").reset_index(name="n_fragments").sort_values(["n_fragments", "scaffold"], ascending=[sparse_end, True], ignore_index=True))
ranked["cumulative"] = ranked["n_fragments"].cumsum()

if by_count:
    selected_scaffolds = ranked.head(SCAFFOLD_NUMBER)
elif SCAFFOLD_BASIS == "fragments":
    # Take whole scaffolds while they fit inside the budget. Always keep 1 at least.
    selected_scaffolds = ranked[ranked["cumulative"] <= SCAFFOLD_FRACTION * len(frame)]
    if selected_scaffolds.empty:
        selected_scaffolds = ranked.head(1)
else:
    # Count scaffolds instead.
    selected_scaffolds = ranked.head(max(1, round(SCAFFOLD_FRACTION * len(ranked))))

# Map back onto the mols: every fragment sitting on one of those scaffolds.
is_selected = scaffolds.isin(set(selected_scaffolds["scaffold"])).to_numpy()

CUT = f"{SCAFFOLD_END} {SCAFFOLD_NUMBER:,}" if by_count else f"{SCAFFOLD_END} {100 * SCAFFOLD_FRACTION:g}% by {SCAFFOLD_BASIS}"

print(f"{len(ranked):,} distinct scaffolds over {int((~acyclic).sum()):,} fragments "
      f"({int(acyclic.sum()):,} acyclic, no scaffold)")

print(f"singleton scaffolds: {int((ranked['n_fragments'] == 1).sum()):,} "
      f"({100 * (ranked['n_fragments'] == 1).mean():.0f}% of scaffolds)")

print(f"{CUT}: {len(selected_scaffolds):,} scaffolds "
      f"({100 * len(selected_scaffolds) / len(ranked):.2f}% of scaffolds) -> {is_selected.sum():,} fragments "
      f"({100 * is_selected.mean():.2f}% of the {len(frame):,} plotted), "
      f"up to {selected_scaffolds['n_fragments'].max():,} fragments per scaffold")

TITLE_SCAFFOLDS = (f"ChEMBL fragment space: the {END_WORD} represented {SELECTION_WORDS}\n"
                   f"{is_selected.sum():,} fragments on {len(selected_scaffolds):,} of those scaffolds, in red")

figure, axes = plt.subplots(figsize=(20, 20))
axes.scatter(embedding[~is_selected, 0], embedding[~is_selected, 1], s=1.0, c="#2a78d6", alpha=0.9, linewidths=0.0, label="Remaining fragments")
# Slightly larger and drawn last, or the selection disappears under the rest.
axes.scatter(embedding[is_selected, 0], embedding[is_selected, 1], s=2.0, c=RED, alpha=0.95, linewidths=0.0, zorder=3, label=f"Fragments on the {END_WORD} represented {SELECTION_WORDS}")

axes.set_title(TITLE_SCAFFOLDS, fontsize=26, loc="left", pad=24)
axes.set_xlabel("UMAP 1", color=GREY, fontsize=20)
axes.set_ylabel("UMAP 2", color=GREY, fontsize=20)
axes.set_xticks([]); axes.set_yticks([])
axes.set_aspect("equal", adjustable="datalim")
legend = axes.legend(fontsize=18, markerscale=8, loc="upper right", frameon=False)
figure.tight_layout()

for extension in ("svg", "png"):
    figure.savefig(FIGURE_STEM.with_name(f"Scaffolds_{SELECTION_LABEL}_{FIGURE_STEM.name}.{extension}"),
                   dpi=200, bbox_inches="tight")
plt.show()

### Clusters within the selected scaffolds

The same scaffold cut from the cell above. HDBSCAN runs on the UMAP coordinates of the selected points only, and the largest few clusters get a colour each; everything else in the selection -- smaller clusters and the points HDBSCAN calls noise -- stays grey, and the rest of the map keeps its blue.

The knobs are at the top of the cell: 
1. `MIN_CLUSTER_SIZE` does what it says 
2. `CLUSTER_SELECTION_METHOD` has 3 options: `"leaf"` cuts the tree into tight, local clusters, `"eom"` tends to merge most of the tail into one blob covering half the map, and `"scaffold"` skips the clustering altogether (it warns, since no HDBSCAN runs) and makes each scaffold in the cut its own cluster, ranked by fragment prevelance. 
3. `N_CLUSTERS_SHOWN` colours that many scaffolds and the rest of the cut stays grey, so `SCAFFOLD_NUMBER = 5` with `N_CLUSTERS_SHOWN = 3` colours three scaffolds and greys the other two. A fraction cut works the same way, just with more scaffolds left over. Nothing is noise in that mode, the legend names the scaffolds instead of numbering clusters, and the figures and CSV get a `_by_scaffold` suffix so they don't overwrite the HDBSCAN ones for the same cut.

`scaffold_clusters` comes back with a row per selected fragment -- molecule, scaffold and cluster number.

**NOTES** 

It's not particularly informative for bottom end. Small clusters capture a lot of noise and are not "more similar". Large clusters look promising but containt chemically distinct moeities. Leaf clustering seemingly "misses" a part of a larger cluster. EOM generates broad. 

*Things considered:*
1. For each scaffold, generate a generic murcko. Group by generic murcko, select 3 largest groups, overlay as clusters. **Problem:** Clusters at loose queries (450k sets) are very top heavy (benzene captures most of scaffolds). At tight queries (80k sets), only 1 cluster produced. 
2. Tanimoto similarity grouping would be the same as grouping by Jaccard, which is what the UMAP does already. 

In [ ]:
import warnings

from sklearn.cluster import HDBSCAN

MIN_CLUSTER_SIZE = 200          # smallest group HDBSCAN allowed to call a cluster
MIN_SAMPLES = None              # None -> MIN_CLUSTER_SIZE; lower = looser, fewer points called noise
CLUSTER_SELECTION_METHOD = "scaffold"   # "leaf" = tight local clusters, "eom" = fewer, much larger ones, "scaffold" = no HDBSCAN, one cluster per scaffold
N_CLUSTERS_SHOWN = 5            
CLUSTER_COLOURS = ["#d1344a", "#2e9e4f", "#c035c0", "#c5c20d", "#07c8d6"]  
OTHER_GREY = "#54534f"          

by_scaffold = CLUSTER_SELECTION_METHOD == "scaffold"
if by_scaffold:
    warnings.warn("CLUSTER_SELECTION_METHOD = 'scaffold': HDBSCAN is bypassed and every scaffold in the cut is its own cluster", stacklevel=2)

selected_points = embedding[is_selected]
selected_scaffolds_per_fragment = scaffolds[is_selected].to_numpy()

if by_scaffold:
    # Rank the scaffolds in the cut by how many fragments they carry, ties broken
    # on the SMILES, and label each fragment with its scaffold's rank.
    order = (pd.Series(selected_scaffolds_per_fragment).value_counts().rename_axis("scaffold").reset_index(name="n_fragments")
             .sort_values(["n_fragments", "scaffold"], ascending=[False, True], ignore_index=True))
    
    label_of = {scaffold: label for label, scaffold in enumerate(order["scaffold"])}
    labels = np.array([label_of[scaffold] for scaffold in selected_scaffolds_per_fragment])
else:
    clusterer = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, cluster_selection_method=CLUSTER_SELECTION_METHOD, copy=True).fit(selected_points)
    labels = clusterer.labels_

scaffold_clusters = pd.DataFrame({"chembl_id": frame.loc[is_selected, "chembl_id"].to_numpy(),
                                  "smiles": frame.loc[is_selected, "smiles"].to_numpy(),
                                  "scaffold": selected_scaffolds_per_fragment,
                                  "umap_1": selected_points[:, 0],
                                  "umap_2": selected_points[:, 1],
                                  "cluster": labels,})          # -1 is HDBSCAN's noise label

# Prominence = how many selected fragments a cluster holds; -1 is never a cluster.
# Sorting the labels first makes the tie-break deterministic.
sizes = scaffold_clusters.loc[scaffold_clusters["cluster"] >= 0, "cluster"].value_counts().sort_index().sort_values(ascending=False, kind="stable")
shown = sizes.head(N_CLUSTERS_SHOWN).index.tolist()
rank = {label: position + 1 for position, label in enumerate(shown)}
scaffold_clusters["cluster_rank"] = scaffold_clusters["cluster"].map(rank).astype("Int64")
scaffold_clusters = scaffold_clusters.sort_values(["cluster_rank", "cluster"], na_position="last", ignore_index=True)

# A scaffold cluster is worth naming by its scaffold; an HDBSCAN one only has its number.
cluster_names = ({label: scaffold for scaffold, label in label_of.items()} if by_scaffold
                 else {label: f"cluster {label}" for label in sizes.index})

# Repeat the palette rather than fall over if more clusters are asked for than there are colours.
colours = [CLUSTER_COLOURS[position % len(CLUSTER_COLOURS)] for position in range(len(shown))]

summary = (scaffold_clusters[scaffold_clusters["cluster_rank"].notna()].groupby(["cluster_rank", "cluster"], observed=True)
           .agg(fragments=("smiles", "size"), scaffolds=("scaffold", "nunique"), umap_1=("umap_1", "mean"), umap_2=("umap_2", "mean"))
           .round(2).assign(colour=colours))

if by_scaffold:
    summary.insert(0, "scaffold", [cluster_names[label] for _, label in summary.index])
#if by_scaffold:
    print(f"{len(sizes):,} scaffolds in the cut, one cluster each, over {is_selected.sum():,} selected fragments")
else:
    print(f"{len(sizes)} clusters over {is_selected.sum():,} selected fragments, "
          f"{int((labels == -1).sum()):,} left as noise "
          f"({100 * (labels == -1).mean():.0f}%)")
print(f"showing the largest {len(shown)}: {summary['fragments'].sum():,} fragments "
      f"({100 * summary['fragments'].sum() / is_selected.sum():.0f}% of the selected set)")

display(summary)

METHOD_WORDS = ("one cluster per scaffold" if by_scaffold else f"{CLUSTER_SELECTION_METHOD} selection")
CLUSTER_LABEL = f"{SELECTION_LABEL}_by_scaffold" if by_scaffold else SELECTION_LABEL

figure, axes = plt.subplots(figsize=(20, 20))
# The whole map first, then the selected scaffolds, then the clusters on top of that.
axes.scatter(embedding[~is_selected, 0], embedding[~is_selected, 1], s=1.0, c="#2a78d6", alpha=0.45, linewidths=0.0, label="Remaining fragments")
other_selected = scaffold_clusters["cluster_rank"].isna().to_numpy()
axes.scatter(scaffold_clusters.loc[other_selected, "umap_1"], scaffold_clusters.loc[other_selected, "umap_2"], s=2.0, c=OTHER_GREY, alpha=0.9, linewidths=0.0, zorder=3,
             label=f"Remaining fragments in the selection ({int(other_selected.sum()):,})")

for colour, label in zip(colours, shown):
    members = scaffold_clusters["cluster"] == label
    name = cluster_names[label]
    axes.scatter(scaffold_clusters.loc[members, "umap_1"], scaffold_clusters.loc[members, "umap_2"], s=7.0, c=colour, alpha=0.95, linewidths=0.0, zorder=4,
                 label=f"{name if len(name) <= 38 else name[:37] + '...'} ({int(members.sum()):,} fragments)")

axes.set_title(f"ChEMBL fragment space: where the {END_WORD} represented {SELECTION_WORDS} "
               f"cluster\n{len(shown)} largest clusters of the "
               f"{is_selected.sum():,} selected fragments, {METHOD_WORDS}", fontsize=24, loc="left", pad=24)
axes.set_xlabel("UMAP 1", color=GREY, fontsize=20)
axes.set_ylabel("UMAP 2", color=GREY, fontsize=20)
axes.set_xticks([]); axes.set_yticks([])
axes.set_aspect("equal", adjustable="datalim")
axes.legend(fontsize=18, markerscale=6, loc="upper right", frameon=False)
figure.tight_layout()

for extension in ("svg", "png"):
    figure.savefig(FIGURE_STEM.with_name(f"Clusters_{CLUSTER_LABEL}_{FIGURE_STEM.name}.{extension}"),
                   dpi=200, bbox_inches="tight")
plt.show()

# What the coloured regions actually are, six mols from each.
examples = scaffold_clusters.dropna(subset=["cluster_rank"]).groupby("cluster", observed=True).head(6)
cluster_mols_grid = Draw.MolsToGridImage([Chem.MolFromSmiles(smiles) for smiles in examples["smiles"]], molsPerRow=6, subImgSize=(200, 160),
                             legends=[f"Cluster {cluster}: {chembl_id}" for cluster, chembl_id in zip(examples["cluster"], examples["chembl_id"])],)

# Save that grid as well for reference
for extension in ("svg", "png"):
    with open(FIGURE_STEM.with_name(f"Clusters_{CLUSTER_LABEL}_{FIGURE_STEM.name}_examples.{extension}"), "wb+") as handle: 
        handle.write(cluster_mols_grid.data)
display(cluster_mols_grid)

# Save the clusters in case mols later overlaid. 
scaffold_clusters.to_csv(DIAGNOSTICS.with_name(f"Clusters_{CLUSTER_LABEL}_{FIGURE_STEM.name}.csv"), index=False)

# Targets behind the scaffold selection

Takes whatever cut the scaffold cell left in `is_selected`, goes back to the release for the targets those fragments have been measured against, and counts them. Two histograms: the `N_TARGETS_SHOWN` most prevalent targets, and the same for ChEMBL's protein classes (level 2 is the "kinase" / "family A GPCR" tier). Both frames, and the fragment -> target map underneath them, are written to `Data/`.

Four notes on these:
1. The frags file holds one row per canonical SMILES, so each fragment carries the ChEMBL id of whichever registration survived dedup. Activities registered against a *different* molregno for the same structure (a salt, or a stereoisomer when `FLATTEN` is on) aren't counted.
2. Bars are distinct fragments per target, not activity rows, so one heavily assayed molecule can't carry a target on its own. Activity rows are kept alongside as `n_activities` in the df.
3. `component_class` files each protein under its *deepest* class only -- the M1 muscarinic receptor is a level-5 "Acetylcholine receptor" and nothing else -- so the cell walks `parent_id` up to `PROTEIN_CLASS_LEVEL` instead of filtering on it. Filtering would drop whole families (every GPCR, most kinases). Targets whose classification never reaches that depth are left out and counted in the print.
4. A target with several components can sit in more than one class, so the class bars don't sum to the fragment total.

In [ ]:
import time

N_TARGETS_SHOWN = 10                 # bars per hist
PROTEIN_CLASS_LEVEL = 3              # ChEMBL class depth: 1 = Enzyme / Membrane rec, 2 = Kinase / Family A GPCR, 3+ narrower
TARGET_TYPES = ("SINGLE PROTEIN",)   # None keeps everything: organisms, cell lines, "Unchecked", etc.
TARGET_LABEL = "gene"                # bar labels: "gene" symbol, "accession" (UniProt) or "name" (ChEMBL pref_name)

# A level is only used if every bar drawn can carry one; a single gap drops the
# whole histogram to the next level down, gene -> accession -> name.
LABEL_ORDER = ("gene", "accession", "name")
LABEL_COLUMN = {"gene": "gene_symbol", "accession": "accession", "name": "target_name"}
if TARGET_LABEL not in LABEL_ORDER:
    raise ValueError(f"TARGET_LABEL must be one of {LABEL_ORDER}, not {TARGET_LABEL!r}")

# is_selected already maps the scaffold cut onto the frags, so this is just the subset.
selected_fragments = frame.loc[is_selected, ["chembl_id", "smiles"]].assign(scaffold=scaffolds[is_selected])

# Fresh read-only connection; the ids go into a temp table.
targets_connection = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
targets_connection.execute("CREATE TEMP TABLE selection (chembl_id TEXT PRIMARY KEY)")
targets_connection.executemany("INSERT OR IGNORE INTO selection VALUES (?)", [(chembl_id,) for chembl_id in selected_fragments["chembl_id"]])

type_filter = "" if TARGET_TYPES is None else f"AND td.target_type IN ({', '.join('?' * len(TARGET_TYPES))})"
type_words = "any target type" if TARGET_TYPES is None else " / ".join(TARGET_TYPES).lower()

# One pass over activities, aggregated to fragment-target pairs so the row count stays low.
print(f"querying {len(selected_fragments):,} fragments against the release ({type_words})...", flush=True)
started = time.time()
fragment_targets = pd.read_sql_query(f"""
    SELECT sel.chembl_id AS fragment_chembl_id, td.tid, td.chembl_id AS target_chembl_id,
           td.pref_name AS target_name, td.target_type, td.organism,
           COUNT(*) AS n_activities
    FROM selection AS sel
    JOIN molecule_dictionary AS md ON md.chembl_id = sel.chembl_id
    JOIN activities AS act ON act.molregno = md.molregno
    JOIN assays AS a ON a.assay_id = act.assay_id
    JOIN target_dictionary AS td ON td.tid = a.tid
    WHERE 1 = 1 {type_filter}
    GROUP BY sel.chembl_id, td.tid
""", targets_connection, params=list(TARGET_TYPES or ()))
print(f"  {len(fragment_targets):,} fragment-target pairs in {time.time() - started:.0f}s")

# component_class only carries each component's deepest class (muscarinic M1 is filed as the level-5 "Acetylcholine receptor"),
# so the tree has to be walked upto the level asked for or whole families go missing.
classification = targets_connection.execute("SELECT protein_class_id, parent_id, pref_name, class_level FROM protein_classification").fetchall()
PARENT = {class_id: parent_id for class_id, parent_id, _, _ in classification}
LEVEL = {class_id: class_level for class_id, _, _, class_level in classification}
CLASS_NAME = {class_id: pref_name for class_id, _, pref_name, _ in classification}

def ancestor(class_id):
    """The PROTEIN_CLASS_LEVEL class this one sits under, or None if it never goes that deep."""
    while class_id is not None and LEVEL[class_id] > PROTEIN_CLASS_LEVEL:
        class_id = PARENT[class_id]
    return CLASS_NAME[class_id] if class_id is not None and LEVEL[class_id] == PROTEIN_CLASS_LEVEL else None

class_map = pd.read_sql_query("""
    SELECT DISTINCT tc.tid, cc.protein_class_id
    FROM target_components AS tc
    JOIN component_class AS cc ON cc.component_id = tc.component_id
""", targets_connection)

# Gene symbol and UniProt accession per target. 
target_labels = pd.read_sql_query("""
    SELECT tc.tid, cs.accession,
           (SELECT csyn.component_synonym FROM component_synonyms AS csyn
             WHERE csyn.component_id = cs.component_id AND csyn.syn_type = 'GENE_SYMBOL'
             ORDER BY csyn.component_synonym LIMIT 1) AS gene_symbol
    FROM target_components AS tc
    JOIN component_sequences AS cs ON cs.component_id = tc.component_id
""", targets_connection)
targets_connection.close()


def joined(values):
    """One label per target: a multi-component target carries all of its symbols."""
    kept = sorted({value for value in values if isinstance(value, str) and value})
    return "/".join(kept) if kept else None


target_labels = target_labels.groupby("tid").agg(gene_symbol=("gene_symbol", joined), accession=("accession", joined))

# Several leaf classes of one target can roll up into the same ancestor, so drop the repeats before the merge or their activities get counted twice.
class_map["protein_class"] = class_map["protein_class_id"].map(ancestor)
class_map = class_map.dropna(subset=["protein_class"]).drop_duplicates(["tid", "protein_class"])

targets = (fragment_targets.groupby(["tid", "target_chembl_id", "target_name", "target_type", "organism"], dropna=False, observed=True)
           .agg(n_fragments=("fragment_chembl_id", "nunique"), n_activities=("n_activities", "sum"))
           .reset_index().merge(target_labels, on="tid", how="left")
           .sort_values(["n_fragments", "n_activities"], ascending=False, ignore_index=True))

target_classes = (fragment_targets.merge(class_map, on="tid").groupby("protein_class", observed=True)
                  .agg(n_fragments=("fragment_chembl_id", "nunique"), n_targets=("tid", "nunique"), n_activities=("n_activities", "sum"))
                  .reset_index().sort_values(["n_fragments", "n_targets"], ascending=False, ignore_index=True))

covered = fragment_targets["fragment_chembl_id"].nunique()
unclassified = fragment_targets.loc[~fragment_targets["tid"].isin(set(class_map["tid"])), "tid"].nunique()

print(f"{covered:,} of the {len(selected_fragments):,} selected fragments have at least one activity "
      f"({100 * covered / len(selected_fragments):.1f}%) over {len(targets):,} targets")
print(f"protein classes at level {PROTEIN_CLASS_LEVEL}: {len(target_classes):,} "
      f"({unclassified:,} of the targets have no class that deep)")

display(targets.head(N_TARGETS_SHOWN))
display(target_classes.head(N_TARGETS_SHOWN))


def histogram(labels, counts, title, name):
    """Horizontal bars, biggest at the top, saved next to the other figures."""
    figure, axes = plt.subplots(figsize=(11, 0.4 * len(labels) + 2.5))
    axes.barh(list(labels)[::-1], list(counts)[::-1], color="#2a78d6")
    axes.set_title(title, fontsize=14, loc="left", pad=12)
    axes.set_xlabel("fragments in the selection", color=GREY)
    axes.tick_params(labelsize=10)
    axes.spines[["top", "right"]].set_visible(False)
    figure.tight_layout()
    for extension in ("svg", "png"):
        figure.savefig(FIGURE_STEM.with_name(f"{name}_{SELECTION_LABEL}_{FIGURE_STEM.name}.{extension}"), dpi=200, bbox_inches="tight")
    plt.show()

top_targets = targets.head(N_TARGETS_SHOWN)

# Walk down from TARGET_LABEL until a level every bar can carry; name always can.
label_level = LABEL_ORDER[-1]
for candidate in LABEL_ORDER[LABEL_ORDER.index(TARGET_LABEL):]:
    missing = int(top_targets[LABEL_COLUMN[candidate]].isna().sum())
    if candidate == LABEL_ORDER[-1] or missing == 0:
        label_level = candidate
        break

    print(f"{missing} of the {len(top_targets)} targets shown have no {candidate}, so the whole histogram falls back")
    
LABEL_WORDS = {"gene": "gene symbols", "accession": "UniProt accessions", "name": "ChEMBL preferred names"}[label_level]
print(f"labelling the target bars with {LABEL_WORDS}")

histogram([f"{label} [{organism}]" if isinstance(organism, str) else str(label) for label, organism in zip(top_targets[LABEL_COLUMN[label_level]], top_targets["organism"])],
          top_targets["n_fragments"], f"Most prevalent targets for the {END_WORD} represented {SELECTION_WORDS}\n"
          f"{covered:,} of {len(selected_fragments):,} selected fragments, {type_words}, labelled by {LABEL_WORDS}", "Targets")

if target_classes.empty:
    print(f"no protein classes at level {PROTEIN_CLASS_LEVEL} for these targets, nothing to plot")
else:
    top_classes = target_classes.head(N_TARGETS_SHOWN)
    histogram(top_classes["protein_class"], top_classes["n_fragments"],
              f"Most prevalent protein classes (level {PROTEIN_CLASS_LEVEL}) for the {END_WORD} represented {SELECTION_WORDS}\n", "TargetClasses")

# Save the two frames + the map they were built from.
for name, frame_out in (("Targets", targets), ("TargetClasses", target_classes), ("Fragment_targets", fragment_targets)):
    path = DIAGNOSTICS.with_name(f"{name}_{SELECTION_LABEL}_{FIGURE_STEM.name}.csv")
    frame_out.to_csv(path, index=False)
    print(f"{len(frame_out):>7,} rows -> {path}")